# Calibração das features de TEXTO — Ambivalência/Hesitância (BAH)

Diagnóstico + ablação do `TextFeaturizer` (A1/A3/A4/H1), no molde do notebook de áudio.
Decide, a grão fino: quais features/léxicos carregam sinal, qual normalização e quais fontes P/N.

**Granularidade:** A1 e a flutuação de A3 são de RESPOSTA (broadcast); A4/H1/A3-entropia são por janela.
Por isso o train é subamostrado **por vídeo** (resposta completa para A1).

Pré-requisito: kernel = `.venv` do projeto; índice de janelas gerado (`mode=preprocess`).

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

def find_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / "configs").is_dir() and (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError("raiz do projeto nao encontrada")
ROOT = find_root(); sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

from src.data.windowing import load_window_index
from src.features.text_features import TextFeaturizer
from src.training.aggregation import calibrate_threshold
from src.training.metrics import video_macro_f1
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- knobs ---
MAX_TRAIN_VIDEOS = 500   # subamostra por video (0 = todos)
SEED = 42
CFG = {}                 # sobreponha flags/params do TextFeaturesConfig aqui, ex.: {"use_emotion": True}

In [ ]:
rng = np.random.default_rng(SEED)
wins = load_window_index(ROOT / "data/interim/windows_index.parquet")
tr = [w for w in wins if w.split == "train" and w.label is not None]
va = [w for w in wins if w.split == "val" and w.label is not None]
if MAX_TRAIN_VIDEOS:
    vids = sorted({w.video_id for w in tr})
    keep = set(rng.choice(vids, size=min(MAX_TRAIN_VIDEOS, len(vids)), replace=False).tolist())
    tr = [w for w in tr if w.video_id in keep]
print(f"train {len(tr)} janelas + val {len(va)} janelas")

ext = TextFeaturizer.from_config(CFG or None, device="auto")
names = ext.feature_names(); print("features:", ext.dim)
Xtr = ext.extract([w.text for w in tr], [w.video_id for w in tr]); ytr = np.array([int(w.label) for w in tr])
Xva = ext.extract([w.text for w in va], [w.video_id for w in va]); yva = np.array([int(w.label) for w in va])
vids_va = np.array([w.video_id for w in va]); vlab = {w.video_id: int(w.video_label) for w in va if w.video_label is not None}

## 1. AUC por feature (val)
Distância de 0.5 = poder discriminativo. Features de RESPOSTA (A1/A3-flutuação) avaliadas contra o
rótulo de VÍDEO (são constantes por vídeo); as de janela contra o rótulo de janela.

In [ ]:
def auc_dir(y, f):
    f = np.nan_to_num(np.asarray(f, float))
    if np.unique(f).size < 2: return 0.5
    try: return float(roc_auc_score(y, f))
    except Exception: return 0.5

# rótulo por vídeo alinhado às janelas de val (p/ features de resposta)
yv_win = np.array([vlab.get(v, 0) for v in vids_va])
resp_pref = ("text_a1_", "text_a3_emotion_fluct", "text_a3_pole", "text_a3_valence")
rows = []
for j, n in enumerate(names):
    y = yv_win if n.startswith(resp_pref) else yva
    a = auc_dir(y, Xva[:, j]); rows.append((n, a, abs(a - 0.5)))
rows.sort(key=lambda r: -r[2])
top = rows[:20]
plt.figure(figsize=(9, 7))
plt.barh([r[0] for r in top][::-1], [r[2] for r in top][::-1], color="#3b7dd8")
plt.xlabel("|AUC - 0.5|"); plt.title("Top-20 features de texto por discriminância (val)")
plt.tight_layout(); plt.show()
for n, a, d in top: print(f"  {n:34s} AUC={a:.3f}")

## 2. Ablação — macro-F1 (vídeo) por grupo, normalização e fonte P/N
RF só nas features de texto; liga/desliga cada grupo (A1/A3/A4/H1), cada normalização e a emoção.

In [ ]:
def rf_macro_f1(Xtr, ytr, Xva, cols):
    if not len(cols): return 0.0
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                min_samples_leaf=2, n_jobs=-1, random_state=SEED).fit(Xtr[:, cols], ytr)
    p = rf.predict_proba(Xva[:, cols])[:, 1]
    _, f1 = calibrate_threshold(p, vids_va, vlab, method="mean_proba", selection="smooth")
    return f1

idx_all = list(range(len(names)))
full = rf_macro_f1(Xtr, ytr, Xva, idx_all)
yv = np.array(list(vlab.values())); base = video_macro_f1(yv, np.full_like(yv, int(round(yv.mean()))))
print(f"TODAS as features: macro-F1 = {full:.4f}  | baseline majoritário = {base:.4f}\n")

groups = {
  "A1 (ambivalência)": [i for i,n in enumerate(names) if n.startswith("text_a1_")],
  "A3 (emoção)":       [i for i,n in enumerate(names) if n.startswith("text_a3_")],
  "A4 (contraste)":    [i for i,n in enumerate(names) if n.startswith("text_a4_")],
  "H1 (hedges)":       [i for i,n in enumerate(names) if n.startswith("text_h1_")],
}
print("Só o grupo (isolado)      |  Sem o grupo (leave-one-out):")
for g, cols in groups.items():
    only = rf_macro_f1(Xtr, ytr, Xva, cols)
    without = rf_macro_f1(Xtr, ytr, Xva, [i for i in idx_all if i not in set(cols)])
    print(f"  {g:22s} só={only:.4f}   sem={without:.4f}  (Δ vs todas={without-full:+.4f})")

## 3. Interpretação
- Grupo com **maior 'só'** = mais forte isolado; **'sem' << todas** = mais insubstituível.
- Compare normalizações editando `CFG={"norm": ["per_word"]}` e re-rodando.
- Decida o léxico final editando as constantes em `src/features/text_features.py` e re-rodando o Passo 1.
- Confirme os finalistas no cross-attention (custa 1 treino Lightning por braço).